# 6.1 · K-Means 聚类 / K-Means Clustering

> **课程定位 / Where this fits**
> 第 1 课，**Part 6 · 无监督学习**。
> Lesson 1, **Part 6 · Unsupervised Learning**.
>
> 前面 Part 4-5 都有"标准答案"（标签）可学。从这里开始进入**无监督**——**没有标签**，要从数据自身的结构里发现规律。
> Parts 4-5 had a "right answer" (labels) to learn from. Here we enter **unsupervised** learning — **no labels**; we must discover structure from the data itself.
>
> 聚类(clustering)是无监督的第一大任务：把相似的样本归到一组。K-Means 是最经典、最常用的聚类算法，也是理解后面所有聚类方法的起点。
> Clustering is the first big unsupervised task: group similar samples together. K-Means is the most classic and widely used clustering algorithm, and the starting point for every method that follows.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $\mathbf{x}_i$ —— 第 $i$ 个样本（$d$ 维向量）/ the $i$-th sample (a $d$-vector)
> - $K$ —— 簇的个数（需事先指定）/ number of clusters (must be set in advance)
> - $C_k$ —— 第 $k$ 个簇（一组样本）/ the $k$-th cluster (a set of samples)
> - $\boldsymbol\mu_k$ —— 第 $k$ 个簇的**质心**（中心点）/ the **centroid** (center) of cluster $k$
> - $\|\cdot\|$ —— 欧氏距离 / Euclidean distance

> 💡 **面试相关 / Interview-relevant**
> - 写出 K-Means 的**目标函数**并说明为什么会收敛（出镜率 ★★★★★）
> - **K-Means++** 解决什么问题（★★★★★）
> - 怎么选簇数 K：**肘部法 / 轮廓系数**（★★★★★）
> - K-Means 的**假设和失效场景**（★★★★★）
> - 为什么聚类前**必须缩放**（★★★★）
>
> Whiteboard hits: objective + why it converges, what K-Means++ fixes, how to pick K, assumptions/failures, why scaling.

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 写出 K-Means 的目标（簇内平方和）并解释 **Lloyd 算法**的两步。
   Write the K-Means objective (within-cluster sum of squares) and explain the two steps of **Lloyd's algorithm**.
2. **从零**实现 K-Means 并与 sklearn 对照。
   Implement K-Means **from scratch** and match sklearn.
3. 解释 **K-Means++** 为什么能改善初始化。
   Explain why **K-Means++** improves initialization.
4. 用**肘部法**和**轮廓系数**选择簇数 K。
   Choose K with the **elbow method** and **silhouette score**.
5. 说清 K-Means 的假设、何时失效、为什么要缩放。
   State K-Means's assumptions, when it fails, and why scaling matters.

## 目录 / TOC
1. [先建直觉：什么是聚类、K-Means 怎么想](#1)
2. [目标函数与 Lloyd 算法 ⭐](#2)
3. [🛍️ 数据：Mall Customers](#3)
4. [从零实现 + 对照 sklearn ⭐](#4)
5. [K-Means++ 与初始化 ⭐](#5)
6. [选 K：肘部 + 轮廓 ⭐](#6)
7. [失效场景与缩放 ⭐](#7)
8. [小结](#8)


<a id="1"></a>
## 1. 先建直觉：什么是聚类、K-Means 怎么想 / Intuition First

**聚类**就是：给一堆没有标签的点，把"长得像"的归成几组。比如商场想把顾客分成几类来做不同的营销，但事先并不知道有哪几类——这正是无监督。
**Clustering** means: given a pile of unlabeled points, group the "similar-looking" ones together. E.g. a mall wants to split customers into types for different marketing, without knowing the types in advance — that's unsupervised.

K-Means 的想法非常朴素，可以用一句话概括：**每个簇用它的中心点（质心）代表，每个样本归到离它最近的质心。**
The idea of K-Means is very simple, in one sentence: **represent each cluster by its center (centroid), and assign each sample to the nearest centroid.**

但这里有个鸡生蛋的小循环（和 GMM 6.6 同源）：
But there's a small chicken-and-egg loop (same flavor as GMM, 6.6):

- 要知道质心在哪，得先知道哪些点属于这个簇（拿它们求平均）。
  To know where a centroid is, you first need to know which points belong to it (average them).
- 要知道点属于哪个簇，得先知道质心在哪（比距离）。
  To know which cluster a point belongs to, you first need the centroids (compare distances).

K-Means 的解法：**先随便放 K 个质心，然后"分配→更新"反复迭代**，直到稳定。下面把它写成精确的目标函数。
K-Means's fix: **drop K centroids somewhere, then alternate "assign → update"** until stable. Next we make this a precise objective.


<a id="2"></a>
## 2. 目标函数与 Lloyd 算法 ⭐ / Objective & Lloyd's Algorithm

K-Means 想最小化的，是**簇内平方和**（within-cluster sum of squares，也叫 inertia）——每个点到它所属质心的距离平方，全部加起来：
What K-Means minimizes is the **within-cluster sum of squares** (WCSS, also called inertia) — the squared distance from each point to its centroid, summed over all points:

$$J = \sum_{k=1}^{K}\ \sum_{\mathbf{x}_i\in C_k}\ \|\mathbf{x}_i - \boldsymbol\mu_k\|^2$$

$J$ 越小，说明簇越"紧"。直接找最优解是 NP 难的，但 **Lloyd 算法**用两步交替逼近它：
Smaller $J$ means tighter clusters. Finding the exact optimum is NP-hard, but **Lloyd's algorithm** approximates it by alternating two steps:

1. **分配步 / Assignment**：固定质心，把每个点归到**最近**的质心所在的簇。
   Fix the centroids; assign each point to the cluster of its **nearest** centroid.
2. **更新步 / Update**：固定分配，把每个质心移到**该簇所有点的均值**处。
   Fix the assignments; move each centroid to the **mean of its cluster's points**.

为什么更新步用"均值"？因为在所有可能的中心点里，**均值正好让到各点的平方距离之和最小**（这是均值的一个基本性质）。
Why does the update step use the "mean"? Because among all possible centers, the **mean is exactly the point minimizing the sum of squared distances** (a basic property of the mean).

**为什么一定收敛**：分配步和更新步**都不会让 $J$ 增大**，而 $J\ge 0$ 有下界——单调不增且有下界，必然收敛。但只保证收敛到**局部**最优，结果依赖初始质心（所以才需要下面的 K-Means++）。
**Why it always converges:** both steps **never increase $J$**, and $J\ge 0$ is bounded below — monotonically non-increasing and bounded, so it must converge. But only to a **local** optimum; the result depends on the initial centroids (hence K-Means++ below).


<a id="3"></a>
## 3. 数据：Mall Customers / The Mall Customers Dataset

我们用聚类教学的经典数据集 **Mall Customers**（商场会员数据）。
We use the classic clustering teaching set **Mall Customers** (shopping-mall membership data).

每行是一位顾客，三个特征：**年龄**、**年收入**（千美元）、**消费分数**（1–100，商场根据消费行为给的评分）。任务是在**没有标签**的情况下发现自然的顾客分群（比如"高收入高消费""低收入谨慎"），用于精准营销。
Each row is a customer with three features: **age**, **annual income** (k$), and **spending score** (1–100, assigned by the mall from spending behavior). The task is to discover natural customer segments (e.g. "high income, high spend" vs "low income, cautious") **without labels**, for targeted marketing.

下面内联一个忠于原数据结构（5 个自然客群）的版本，并先画出来认识一下。
We inline a version faithful to the original structure (5 natural segments) and plot it first.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
np.set_printoptions(precision=3, suppress=True)

def make_mall(seed=0):
    rng = np.random.default_rng(seed)
    # 5 个真实客群: ((年收入k$, 消费分), 人数, (年龄下限,上限))
    groups = [((55, 50), 120, (25, 60)),   # 中等收入中等消费(主流人群)
              ((25, 80), 35,  (18, 35)),   # 低收入高消费(年轻冲动型)
              ((85, 82), 40,  (28, 42)),   # 高收入高消费(理想目标客户)
              ((85, 18), 38,  (35, 60)),   # 高收入低消费(谨慎的富人)
              ((26, 18), 35,  (40, 68))]   # 低收入低消费
    rows = []
    for (inc, spd), n, (amin, amax) in groups:
        income = rng.normal(inc, 8, n).clip(15, 140)
        spend = rng.normal(spd, 9, n).clip(1, 99)
        age = rng.integers(amin, amax, n)
        rows.append(np.c_[age, income, spend])
    X = np.vstack(rows); rng.shuffle(X)
    return pd.DataFrame(X, columns=["age", "income_k", "spending"])

mall = make_mall()
print(f"Mall Customers: {mall.shape} 位顾客 / customers")
print(mall.describe().round(1).to_string())

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(mall["income_k"], mall["spending"], s=20, alpha=0.6)
ax.set_xlabel("年收入 annual income (k$)")
ax.set_ylabel("消费分 spending score (1-100)")
ax.set_title("Mall Customers: 肉眼可见若干自然客群(待聚类发现) / natural segments to discover")
plt.tight_layout(); plt.show()


<a id="4"></a>
## 4. 从零实现 + 对照 sklearn ⭐ / From Scratch & vs sklearn

我们先**缩放**特征（第 7 节会解释为什么必须缩放），然后在 income/spending 两维上聚类，方便可视化。代码就是第 2 节两步的直译。
We first **scale** the features (Section 7 explains why this is mandatory), then cluster on the income/spending plane for easy visualization. The code is a direct translation of the two steps in Section 2.


In [ ]:
from sklearn.preprocessing import StandardScaler
X2 = mall[["income_k", "spending"]].values
Xs = StandardScaler().fit_transform(X2)      # 标准化 (3.4): 每列减均值除标准差

def kmeans_scratch(X, K, n_iter=100, seed=0):
    rng = np.random.default_rng(seed)
    # 从所有样本里随机挑 K 个点(不重复)当初始质心; X[索引数组] 一次取出多行
    mu = X[rng.choice(len(X), K, replace=False)]
    for _ in range(n_iter):
        # --- 分配步 / assignment ---
        # X[:,None,:] 形状变 (n,1,d), mu[None,:,:] 形状变 (1,K,d); 两者相减触发"广播",
        # 得到 (n,K,d) = 每个点到每个质心在每一维上的差; 平方后对最后一维(d)求和 → (n,K) 距离²
        d = ((X[:, None, :] - mu[None, :, :]) ** 2).sum(-1)
        labels = d.argmin(1)                  # 对每一行(每个点)取距离最小的质心编号 → 该点的簇
        # --- 更新步 / update ---
        # 对每个簇 k, 取出属于它的点 X[labels==k] 求列均值当新质心;
        # 若某簇暂时没有点(空簇), 就保留旧质心 mu[k] 避免出错
        new_mu = np.array([X[labels == k].mean(0) if (labels == k).any() else mu[k]
                           for k in range(K)])
        if np.allclose(new_mu, mu):           # 质心几乎不再移动 → 已收敛, 提前结束
            break
        mu = new_mu
    # 目标函数 J: mu[labels] 把每个点替换成它所属质心, 再算到该质心的平方距离并求和
    inertia = ((X - mu[labels]) ** 2).sum()
    return labels, mu, inertia

labels, mu, inertia = kmeans_scratch(Xs, K=5)
print(f"从零 K-Means inertia (簇内平方和 J): {inertia:.1f}")

from sklearn.cluster import KMeans
km = KMeans(n_clusters=5, n_init=10, random_state=0).fit(Xs)
print(f"sklearn K-Means inertia: {km.inertia_:.1f}  (与从零一致 / matches)")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(Xs[:, 0], Xs[:, 1], c=labels, cmap="tab10", s=20, alpha=0.7)
ax.scatter(mu[:, 0], mu[:, 1], c="black", marker="X", s=200, label="质心 centroid")
ax.set_xlabel("income (标准化 std)"); ax.set_ylabel("spending (标准化 std)"); ax.legend()
ax.set_title("从零 K-Means (K=5): 发现 5 个客群 / found 5 segments")
plt.tight_layout(); plt.show()


<a id="5"></a>
## 5. K-Means++ 与初始化 ⭐ / K-Means++ and Initialization

第 2 节说过 Lloyd 只收敛到**局部**最优，结果取决于初始质心。如果一开始随便撒，可能两个质心落在同一团里，最后分得很糟。
Section 2 noted Lloyd only reaches a **local** optimum, depending on the initial centroids. Random placement can drop two centroids in the same blob, giving a bad final result.

**K-Means++** 是一种聪明的播种方式：第一个质心随机选；之后每个新质心，**以正比于"到已选质心距离平方"的概率**来选——也就是倾向于选离现有质心**远**的点。这样初始质心天然分散，极大降低落入坏局部最优的概率。它是 sklearn 的默认 (`init='k-means++'`)。
**K-Means++** is a smart seeding scheme: pick the first centroid at random; then pick each new centroid **with probability proportional to its squared distance to the already-chosen centroids** — i.e. favor points **far** from existing centroids. This spreads the initial centroids out and greatly reduces bad local optima. It is sklearn's default (`init='k-means++'`).

另外 `n_init` 表示"用不同初始化跑很多次、取 inertia 最小的那次"，是另一层保险。
Also `n_init` means "run several times with different inits and keep the lowest-inertia one" — a second safety net.

下面对比"随机初始化"和"K-Means++"各跑 30 次的 inertia 分布。
Below we compare the inertia distribution of 30 runs each, random init vs K-Means++.


In [ ]:
rand_inertias = [KMeans(5, init="random", n_init=1, random_state=s).fit(Xs).inertia_ for s in range(30)]
pp_inertias   = [KMeans(5, init="k-means++", n_init=1, random_state=s).fit(Xs).inertia_ for s in range(30)]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(rand_inertias, bins=15, alpha=0.6, label="随机初始化 random init")
ax.hist(pp_inertias, bins=15, alpha=0.6, label="k-means++")
ax.set_xlabel("inertia (越低越好 / lower=better)"); ax.set_ylabel("频次 frequency"); ax.legend()
ax.set_title("K-Means++ 初始化: inertia 更低更稳定(更少陷坏局部最优)")
plt.tight_layout(); plt.show()
print(f"random init  inertia: 均值 mean {np.mean(rand_inertias):.1f}, 最差 worst {max(rand_inertias):.1f}")
print(f"k-means++    inertia: 均值 mean {np.mean(pp_inertias):.1f}, 最差 worst {max(pp_inertias):.1f}")
print("k-means++ 几乎每次都收敛到好解 / k-means++ almost always reaches a good solution")


<a id="6"></a>
## 6. 选 K：肘部 + 轮廓 ⭐ / Choosing K: Elbow + Silhouette

K 要事先指定，但没有标签时怎么知道几个簇合适？两个常用工具：
K must be set in advance, but with no labels how do we know the right number? Two common tools:

**肘部法 / Elbow method**：画 inertia 随 K 的变化曲线。inertia 一定随 K 增大而下降（簇越多越紧），但在"真实簇数"之后，下降会突然变缓——曲线出现一个**拐点（肘部）**，那就是好的 K。
Plot inertia versus K. Inertia always decreases as K grows (more clusters fit tighter), but past the "true" number the decrease suddenly flattens — the curve has an **elbow**, and that's a good K.

**轮廓系数 / Silhouette score**：对每个点算 $s = \dfrac{b - a}{\max(a, b)}$，其中 $a$ 是它到**同簇**其他点的平均距离，$b$ 是它到**最近的邻簇**所有点的平均距离。$s\in[-1,1]$，**越大越好**（点离自己簇近、离别的簇远）。取平均轮廓最大的 K，比肘部更客观。
For each point compute $s = \dfrac{b - a}{\max(a, b)}$, where $a$ is its mean distance to **same-cluster** points and $b$ its mean distance to the **nearest other cluster**. $s\in[-1,1]$, **higher is better** (close to own cluster, far from others). Pick the K with the largest mean silhouette — more objective than the elbow.


In [ ]:
from sklearn.metrics import silhouette_score
Ks = range(2, 11)
inertias = [KMeans(k, n_init=10, random_state=0).fit(Xs).inertia_ for k in Ks]
sils = [silhouette_score(Xs, KMeans(k, n_init=10, random_state=0).fit_predict(Xs)) for k in Ks]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(list(Ks), inertias, "o-"); axes[0].axvline(5, color="r", ls="--", label="肘部 elbow ≈ 5")
axes[0].set_xlabel("K"); axes[0].set_ylabel("inertia"); axes[0].legend(); axes[0].set_title("肘部法 elbow method")
best_k = list(Ks)[int(np.argmax(sils))]
axes[1].plot(list(Ks), sils, "s-")
axes[1].axvline(best_k, color="r", ls="--", label=f"最佳 best K={best_k}")
axes[1].set_xlabel("K"); axes[1].set_ylabel("平均轮廓系数 mean silhouette")
axes[1].legend(); axes[1].set_title("轮廓系数 silhouette")
plt.tight_layout(); plt.show()
print(f"肘部≈5(我们造数据时就是5群); 轮廓系数最佳 K={best_k}")
print(f"Elbow ≈ 5 (we generated 5 groups); silhouette best K={best_k}")


<a id="7"></a>
## 7. 失效场景与缩放 ⭐ / When K-Means Fails, and Scaling

K-Means 只用到"到质心的欧氏距离"和"求均值"，这背后藏着几个**假设**：簇是**球形、大小相近、密度相近**。一旦违反就会失败：
K-Means uses only "Euclidean distance to centroid" and "the mean", which hides several **assumptions**: clusters are **spherical, similar in size, similar in density**. When violated, it fails:

- **非球形簇**（月牙、环形）：K-Means 只能用直线/球形切分，切不出弯曲的簇 → 用 DBSCAN(6.4) 或谱聚类(6.7)。
  **Non-spherical clusters** (moons, rings): K-Means can only cut with straight/spherical boundaries → use DBSCAN (6.4) or spectral clustering (6.7).
- **大小/密度差异大**：大簇会"吞掉"邻近的小簇。
  **Very different sizes/densities:** a big cluster can swallow a nearby small one.
- **不缩放**：K-Means 靠距离，量纲大的特征（如收入 0–140 vs 年龄 18–68）会主导距离，小量纲特征几乎被忽略 → **聚类前必须标准化**（和 KNN/SVM 一样，见 5.3/5.5）。
  **No scaling:** distance is dominated by large-scale features (income 0–140 vs age 18–68), drowning out small ones → **always standardize before clustering** (like KNN/SVM, 5.3/5.5).

下面两张图分别展示"月牙失败"和"不缩放被大量纲特征绑架"。
The two plots below show the "moons failure" and the "hijacked by the large-scale feature when unscaled".


In [ ]:
from sklearn.datasets import make_moons
Xm, _ = make_moons(300, noise=0.06, random_state=0)
km_moon = KMeans(2, n_init=10, random_state=0).fit_predict(Xm)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].scatter(Xm[:, 0], Xm[:, 1], c=km_moon, cmap="coolwarm", s=15)
axes[0].set_title("月牙形上失败 / fails on moons\n(只能切球形, 切不出弯月)")

# 不缩放: age(18-68) vs income(15-140) 量纲差异大 / unscaled, mismatched scales
X_age = mall[["age", "income_k"]].values
lab_unscaled = KMeans(5, n_init=10, random_state=0).fit_predict(X_age)
axes[1].scatter(X_age[:, 0], X_age[:, 1], c=lab_unscaled, cmap="tab10", s=15)
axes[1].set_xlabel("age"); axes[1].set_ylabel("income_k")
axes[1].set_title("不缩放: 簇沿大量纲(income)切, age 几乎被忽略\nunscaled: split by income, age ignored")
plt.tight_layout(); plt.show()
print("月牙→需 DBSCAN/谱聚类; 量纲不一→聚类前必须标准化")
print("Moons → use DBSCAN/spectral; mismatched scales → standardize before clustering.")


<a id="8"></a>
## 8. 小结 / Summary

```
目标 / objective: 最小化簇内平方和 J = Σ_k Σ_{x∈C_k} ‖x-μ_k‖²
Lloyd 算法: 分配(归到最近质心) ↔ 更新(质心=簇均值), 交替到收敛
收敛: 两步都不增 J 且 J≥0 → 单调收敛到局部最优(依赖初始化)
K-Means++: 按距离²概率分散播种 → 避免坏局部最优(默认); n_init 多次重启
选 K: 肘部法(inertia 拐点) + 轮廓系数(越高越好, 更客观)
假设: 球形/等大小/等密度 + 欧氏距离 → 月牙/环形失败; 必须缩放
硬分配(一个点只属一个簇) — 对比 GMM(6.6) 的软分配
```

### 💡 面试速查 / Interview cheat-sheet
1. **目标=簇内平方和(inertia)**；Lloyd "分配+更新"交替，收敛到局部最优。
   Objective = WCSS (inertia); Lloyd alternates assign+update, converging to a local optimum.
2. **K-Means++** 分散初始质心、避免坏局部最优（默认）。
   K-Means++ spreads initial centroids to avoid bad local optima (default).
3. **选 K**：肘部(拐点) + 轮廓(最大)；业务也可直接定 K。
   Choose K via elbow (kink) + silhouette (max); business may fix K directly.
4. **假设球形/等密度簇** → 月牙/环形失败 → 用 DBSCAN/谱聚类。
   Assumes spherical/equal-density clusters → fails on moons/rings → DBSCAN/spectral.
5. **必须缩放**（距离对量纲敏感）；K-Means 是**硬分配**(对比 GMM 软分配)。
   Must scale (distance is scale-sensitive); K-Means is hard assignment (vs GMM's soft).

### 下一节 / Next
**6.2 Mini-batch K-Means**——数据上百万时，用小批近似质心更新，省内存、可流式。
**6.2 Mini-batch K-Means** — for millions of points, approximate centroid updates from small batches: constant memory, streamable.
